# 06c — Optuna Hyperparameter Tuning (LightGBM LambdaRank)

Tune key LightGBM params on the valid split.
Objective: maximize NDCG@10 (primary) while keeping Recall@10 ≥ baseline.

Requires: `pip install optuna`

**Inputs**: `cache_drive/train_matrix_valid.parquet`, `cache_drive/valid_gt.parquet`
**Outputs**: `cache_drive/best_params.json`, `cache_drive/model_tuned.txt`

In [ ]:
# Local kernel: assume deps already installed.
# pip install optuna lightgbm pyarrow pandas numpy
print('Skipping pip install (local kernel).')

In [ ]:
import os, sys

PROJECT_DIR  = r'd:/Datathon_2'
TRAINING_DIR = os.path.join(PROJECT_DIR, 'training')
CACHE_DIR    = os.path.join(PROJECT_DIR, 'cache_drive')

if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print('CACHE_DIR:', CACHE_DIR)

In [ ]:
VALID_START = '2026-03-13'

# Optuna budget
N_TRIALS    = 50    # increase to 100 if time allows
N_ROUNDS    = 1000  # max boosting rounds per trial (early stopping at 80)
EARLY_STOP  = 80

print(f'N_TRIALS={N_TRIALS}, N_ROUNDS={N_ROUNDS}')

In [ ]:
import time
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from utils.metrics import mean_recall_at_k, mean_ndcg_at_k

t0 = time.time()
full = pd.read_parquet(f'{CACHE_DIR}/train_matrix_valid.parquet')
print(f'full: {full.shape} | {time.time()-t0:.1f}s')

DROP = {'user_id', 'item_id', 'label', 'title', 'posted_date',
        'expected_expired_date', 'first_evt_date', 'last_evt_date',
        'last_snap_date', 'project_id'}

# Optionally drop SHAP-identified noise features
noise_path = f'{CACHE_DIR}/shap_noise_features.json'
if os.path.exists(noise_path):
    with open(noise_path) as f:
        noise_feats = set(json.load(f))
    print(f'Dropping {len(noise_feats)} SHAP noise features')
    DROP = DROP | noise_feats
else:
    print('No SHAP noise file found — using all features')

cat_cols = [c for c in full.columns if c not in DROP and full[c].dtype == 'object']
num_cols = [c for c in full.columns if c not in DROP
            and full[c].dtype != 'object' and full[c].dtype != 'datetime64[ns]']
for c in cat_cols:
    full[c] = full[c].astype('category')
feat_cols = cat_cols + num_cols
print(f'#features: {len(feat_cols)}')

In [ ]:
from hashlib import md5

def _hash01(s):
    return int(md5(str(s).encode()).hexdigest(), 16) % 100

full['_h'] = full['user_id'].map(_hash01)
tr_mask = full['_h'] < 80
va_mask  = full['_h'] >= 80

X_tr = full.loc[tr_mask, feat_cols]
y_tr = full.loc[tr_mask, 'label'].values
g_tr = full.loc[tr_mask].groupby('user_id', sort=False).size().values

X_va = full.loc[va_mask, feat_cols]
y_va = full.loc[va_mask, 'label'].values
g_va = full.loc[va_mask].groupby('user_id', sort=False).size().values

valid_gt = pd.read_parquet(f'{CACHE_DIR}/valid_gt.parquet')
gt_full  = valid_gt.groupby('user_id')['item_id'].agg(set).to_dict()

print(f'train: {len(X_tr):,} | valid: {len(X_va):,}')

In [ ]:
MONO_MAP = {
    'age_at_train_end':         -1,
    'recency_evt_days':         -1,
    'u_recency_days':           -1,
    'i_CR_30d':                  1,
    'i_contacts_24h_mean_30d':   1,
    'i_n_pos_30d':               1,
    'i_n_pos_7d':                1,
    'u_n_pos_30d':               1,
    'i_velocity_contacts':       1,
    'i_velocity_views':          1,
    'x_price_affinity':          1,
}
mono_list = [MONO_MAP.get(c, 0) for c in feat_cols]
print(f'Monotonic active: {sum(v!=0 for v in mono_list)}')

In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = dict(
        objective='lambdarank',
        metric='ndcg',
        ndcg_eval_at=[10],
        verbosity=-1,
        seed=42,
        monotone_constraints=mono_list,
        monotone_constraints_method='advanced',
        # ---- tuned params ----
        learning_rate=trial.suggest_float('learning_rate', 0.02, 0.15, log=True),
        num_leaves=trial.suggest_int('num_leaves', 63, 511),
        min_data_in_leaf=trial.suggest_int('min_data_in_leaf', 50, 500),
        feature_fraction=trial.suggest_float('feature_fraction', 0.5, 1.0),
        bagging_fraction=trial.suggest_float('bagging_fraction', 0.5, 1.0),
        bagging_freq=trial.suggest_int('bagging_freq', 1, 10),
        lambda_l1=trial.suggest_float('lambda_l1', 0.0, 5.0),
        lambda_l2=trial.suggest_float('lambda_l2', 0.0, 5.0),
        max_bin=trial.suggest_categorical('max_bin', [127, 255, 511]),
    )
    d_tr = lgb.Dataset(X_tr, label=y_tr, group=g_tr,
                       categorical_feature=cat_cols, free_raw_data=False)
    d_va = lgb.Dataset(X_va, label=y_va, group=g_va,
                       categorical_feature=cat_cols, reference=d_tr,
                       free_raw_data=False)
    cb = [lgb.early_stopping(EARLY_STOP, verbose=False),
          lgb.log_evaluation(-1)]
    m = lgb.train(params, d_tr, num_boost_round=N_ROUNDS,
                  valid_sets=[d_va], valid_names=['valid'], callbacks=cb)

    preds_dict = {}
    full_va = full[va_mask].copy()
    full_va['_pred'] = m.predict(X_va, num_iteration=m.best_iteration)
    for uid, grp in full_va.groupby('user_id', sort=False):
        preds_dict[uid] = grp.nlargest(10, '_pred')['item_id'].tolist()
    gt_va = {u: gt_full[u] for u in preds_dict if u in gt_full}

    ndcg = mean_ndcg_at_k(preds_dict, gt_va, k=10)
    recall = mean_recall_at_k(preds_dict, gt_va, k=10)

    trial.set_user_attr('recall_at_10', recall)
    trial.set_user_attr('best_iter', m.best_iteration)
    return ndcg

print('Objective defined.')

In [ ]:
study = optuna.create_study(direction='maximize',
                            study_name='lgbm_lambdarank',
                            sampler=optuna.samplers.TPESampler(seed=42))

# Seed with known-good default params
study.enqueue_trial({
    'learning_rate': 0.05, 'num_leaves': 127, 'min_data_in_leaf': 200,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'lambda_l1': 0.0, 'lambda_l2': 1.0, 'max_bin': 255,
})

t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'\nDone in {(time.time()-t0)/60:.1f} min')

bt = study.best_trial
print(f'Best NDCG@10: {bt.value:.4f}')
print(f'Best Recall@10: {bt.user_attrs["recall_at_10"]:.4f}')
print('Best params:', bt.params)

In [ ]:
# Retrain final model with best params + full train budget
best_params = study.best_trial.params
best_iter   = study.best_trial.user_attrs['best_iter']

final_params = dict(
    objective='lambdarank', metric='ndcg', ndcg_eval_at=[10],
    verbosity=-1, seed=42,
    monotone_constraints=mono_list,
    monotone_constraints_method='advanced',
    **best_params,
)

d_full = lgb.Dataset(full[feat_cols], label=full['label'].values,
                     group=full.groupby('user_id', sort=False).size().values,
                     categorical_feature=cat_cols, free_raw_data=False)

t0 = time.time()
model_tuned = lgb.train(final_params, d_full,
                         num_boost_round=int(best_iter * 1.1))  # +10% rounds on full data
print(f'Retrained in {time.time()-t0:.0f}s | rounds: {model_tuned.num_trees()}')
model_tuned.save_model(f'{CACHE_DIR}/model_tuned.txt')

with open(f'{CACHE_DIR}/best_params.json', 'w') as f:
    json.dump({'params': best_params, 'best_iter': best_iter,
               'ndcg_at_10': bt.value,
               'recall_at_10': bt.user_attrs['recall_at_10']}, f, indent=2)
print('Saved model_tuned.txt + best_params.json')

In [ ]:
# Summary: compare baseline vs tuned on valid split
import os

results = []
for label, model_path in [('baseline', 'model.txt'), ('tuned', 'model_tuned.txt')]:
    mp = f'{CACHE_DIR}/{model_path}'
    if not os.path.exists(mp):
        continue
    m = lgb.Booster(model_file=mp)
    full_va = full[va_mask].copy()
    full_va['_pred'] = m.predict(X_va)
    preds_dict = {}
    for uid, grp in full_va.groupby('user_id', sort=False):
        preds_dict[uid] = grp.nlargest(10, '_pred')['item_id'].tolist()
    gt_va = {u: gt_full[u] for u in preds_dict if u in gt_full}
    r10 = mean_recall_at_k(preds_dict, gt_va, k=10)
    n10 = mean_ndcg_at_k(preds_dict, gt_va, k=10)
    results.append({'model': label, 'recall@10': r10, 'ndcg@10': n10})
    print(f'{label:10s}  Recall@10={r10:.4f}  NDCG@10={n10:.4f}')

pd.DataFrame(results).to_csv(f'{CACHE_DIR}/hpt_comparison.csv', index=False)